In [ ]:
import numpy as np
import pandas as pd
import os
from scipy.stats import sem
import scipy

## Configuration

In [ ]:
dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"] # DTD | EuroSAT | GTSRB | MNIST | RESISC45 | Stanford_Cars | SUN397 | SVHN
domain = "Base_Fine_Tuned" # Base_Fine_Tuned | Fine_Tuned_Layer_Skipping
domain_type = "Reverse_Probe"
model_name = "CLIP_ViT_Vision" # DeiT | CLIP_ViT_Vision | Google_ViT
transformation = ["Standard", "Base_Fine_Tuned_Classifier", "Base_Linear_Probe"] # "Standard" | "Base_Fine_Tuned_Classifier" | "Base_Linear_Probe"
indices = [i for i in range(12)]
size = [i for i in range(1,6)]

## Loading Data

In [ ]:
def find_best_acc(data):
    acc = {i: [] for i in indices} # Indices, then len(data)
    for i in indices:
        for j in range(len(data)):
            acc[i].append(data[j]["Classification_Accuracy"][i])

    best_acc = {}

    for i in acc:
        arr = acc[i]
        max_val = max(arr)
        best_acc[i] = max_val

    final = []
    for i in indices:
        final.append(best_acc[i])

    best_accuracy = max(final)
    index = final.index(best_accuracy)

    return best_accuracy, index

In [ ]:
big_data = {i: {} for i in range(len(dataset_name))}

for k in range(len(dataset_name)):
    said=k
    results_path = f"../Data/{domain_type}/{dataset_name[said]}/{domain}" # /Entire_Transformation_Matrix_W"
    results = {}

    for i in size:
        results[i] = []
        path = f"{results_path}/{i}/Entire_Transformation_Matrix_W"
        try:
            for filename in os.listdir(path):
                if filename in [f"Base_Fine_Tuned_Classifier_Results_{i}.json"]: 
                    file_path = os.path.join(path, filename)
                    if os.path.isfile(file_path):
                        results[i].append(file_path)
        except FileNotFoundError:
            print(f"Error: The Folder '{path}' was not found.")
        except Exception as e:
            print(f"An error occured: {e}")

        data = [pd.read_json(i) for i in results[i]]
        results[i] = data
        big_data = results[i]
    
    best_acc = []
    index = []
    print(f"{model_name} - {dataset_name[said]}: Ablation Base with Fine-Tuned Classifier")
    for i in size:
        acc, ind = find_best_acc(results[i])
        best_acc.append(acc)
        index.append(ind)
        print(f"Set {i} Best Accuracy: {best_acc[i-1]} | Transformation Layer (0-11): {index[i-1]}")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(best_acc)-1, loc=np.mean(best_acc), scale=sem(best_acc))
    print(f"Average: {np.mean(best_acc)}, Error: +- {ci_high-np.mean(best_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}")